# Lab 8 — Structured Streaming

### O mesmo DataFrame, agora com dados que não param de chegar

Até aqui processamos dados **parados**: arquivos que já existiam. Em muitos casos os dados chegam continuamente (cliques, pedidos, sensores, logs) e o valor está em reagir rápido.

O Structured Streaming usa **a mesma API de DataFrames** que já conhecemos. A diferença está em como a consulta é executada.

**Objetivos**
- Entender o modelo de tabela ilimitada e o micro-batch
- Conhecer sources, sinks, output modes e triggers
- Agregar por **tempo do evento** com janelas
- Tratar dados atrasados com **watermark**
- Garantir tolerância a falhas com **checkpoint**
- Usar `availableNow` para ingestão incremental

---
## 1. Teoria

### Batch vs streaming

| | Batch | Streaming |
|-|-------|-----------|
| Dados | Conjunto fechado | Fluxo sem fim |
| Execução | Roda uma vez e termina | Roda continuamente |
| Leitura | `spark.read` | `spark.readStream` |
| Escrita | `df.write` | `df.writeStream` |
| Latência típica | Minutos a horas | Segundos |

### O modelo de tabela ilimitada

O Spark trata o fluxo como uma **tabela que cresce sem parar**. Cada novo evento é uma nova linha. Você escreve a consulta como se fosse sobre a tabela inteira, e o Spark a executa **de forma incremental**: processa só o que chegou e atualiza o resultado.

```
   tempo ───────────────────────────────────────────────────▶

   fonte      [e1 e2]        [e3 e4 e5]        [e6]
                 │               │               │
   tabela     ┌──────┐       ┌──────┐       ┌──────┐
   de entrada │e1 e2 │       │e1 .. │       │e1 .. │
              └──────┘       │   e5 │       │   e6 │
                 │           └──────┘       └──────┘
   consulta   groupBy        groupBy        groupBy     <- mesma consulta
                 │               │               │
   resultado  [A:2]          [A:3 B:2]      [A:3 B:3]   <- atualizado a cada micro-batch
```

### Componentes de uma consulta de streaming

```
   SOURCE ──▶ TRANSFORMAÇÕES ──▶ SINK
   (Kafka,     (select, filter,    (arquivos, Kafka,
    arquivos,   groupBy, join,      Delta, memória,
    rate)       janelas)            foreachBatch)
                    │
              CHECKPOINT: offsets processados + estado das agregações
              TRIGGER   : quando rodar cada micro-batch
```

### Output modes

| Modo | O que é escrito em cada micro-batch | Uso típico |
|------|-------------------------------------|-----------|
| `append` | Só linhas novas que **não mudarão mais** | Eventos sem agregação; janelas fechadas pelo watermark |
| `update` | Só linhas do resultado que **mudaram** | Dashboards, upserts |
| `complete` | O resultado **inteiro**, de novo | Agregações pequenas |

### Event time vs processing time

- **Processing time:** quando o Spark recebeu o evento.
- **Event time:** quando o evento aconteceu (um campo dentro do dado).

Um pedido feito às 10:01 num celular sem sinal pode chegar às 10:20. Para métricas de negócio, quase sempre queremos **event time**. Isso exige lidar com **dados atrasados**, que é o papel do watermark (seção 5).

---
## 2. Setup

In [ ]:
# !pip install -q pyspark==4.0.1

In [ ]:
import os, json, time, shutil, random, uuid
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = (
    SparkSession.builder
    .appName("Aula 6 - Structured Streaming")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")     # cada partição de shuffle vira estado; poucas para a aula
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.sql.session.timeZone", "UTC")    # horários dos eventos e do watermark no mesmo fuso
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

BASE = "/tmp/aula6"
shutil.rmtree(BASE, ignore_errors=True)
os.makedirs(BASE)

---
## 3. Primeiro stream: a fonte `rate`

A fonte `rate` gera `N` linhas por segundo com as colunas `timestamp` e `value`. É útil para testes, porque não depende de nenhum sistema externo.

O sink `memory` guarda o resultado em uma tabela temporária consultável com SQL. Use-o **só para aprendizado e debug**, porque tudo fica na memória do driver.

In [ ]:
rate = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 20)
    .load()
)
print("É streaming?", rate.isStreaming)
rate.printSchema()

In [ ]:
events = (
    rate
    .withColumn("category", F.element_at(F.array(F.lit("Audio"), F.lit("Storage"), F.lit("Cables")),
                                         (F.col("value") % 3 + 1).cast("int")))
    .withColumn("amount", F.round(F.rand() * 100, 2))
)

q_rate = (
    events.writeStream
    .format("memory")
    .queryName("rate_events")          # nome da tabela em memória
    .outputMode("append")
    .trigger(processingTime="1 second")
    .start()
)

q_rate.awaitTermination(5)             # deixa rodar ~5 segundos (a célula fica bloqueada)
spark.sql("SELECT category, COUNT(*) AS n, ROUND(SUM(amount), 2) AS total FROM rate_events GROUP BY category").show()

In [ ]:
# A consulta continua rodando em segundo plano. Monitoramento:
print("Ativa?     ", q_rate.isActive)
print("Status:    ", q_rate.status)

p = q_rate.lastProgress
print("Último batch:", p["batchId"], "| linhas:", p["numInputRows"],
      "| linhas/s processadas:", round(p["processedRowsPerSecond"] or 0, 1))

In [ ]:
q_rate.stop()
print("Ativa?", q_rate.isActive)

> Sempre pare as consultas que não usa mais. Uma consulta esquecida continua consumindo CPU e memória. `spark.streams.active` lista todas as ativas.

---
## 4. Fonte de arquivos: pedidos chegando em uma pasta

Agora um cenário mais realista: um sistema deposita arquivos JSON com eventos de pedidos em uma pasta. Cada arquivo novo é processado uma única vez.

Dois cuidados importantes com fontes de arquivo:
- **O schema é obrigatório.** Em streaming o Spark não infere schema por padrão, porque ele precisa ser estável durante toda a vida da consulta.
- **Os arquivos devem aparecer completos.** Se o Spark ler um arquivo enquanto ele ainda está sendo escrito, lê dados parciais. O padrão é escrever em outra pasta e depois **mover** (mover é atômico no mesmo sistema de arquivos).

In [ ]:
INPUT   = f"{BASE}/input"
STAGING = f"{BASE}/staging"
os.makedirs(INPUT); os.makedirs(STAGING)

REGIONS    = ["North", "Northeast", "Midwest", "Southeast", "South"]
CATEGORIES = ["Computers", "Peripherals", "Audio", "Storage", "Cables"]
rng = random.Random(7)

def emit(events, folder=INPUT):
    """Escreve uma lista de eventos em STAGING e move para a pasta de entrada (escrita atômica)."""
    os.makedirs(folder, exist_ok=True)
    name = f"events_{uuid.uuid4().hex[:8]}.json"
    tmp = f"{STAGING}/{name}"
    with open(tmp, "w") as f:
        for e in events:
            f.write(json.dumps(e) + "\n")
    os.replace(tmp, f"{folder}/{name}")
    return len(events)

def make_event(event_time, region=None, category=None, amount=None):
    return {
        "event_id":   uuid.uuid4().hex,
        "event_time": event_time.strftime("%Y-%m-%d %H:%M:%S"),
        "order_id":   rng.randint(1, 10_000),
        "region":     region or rng.choice(REGIONS),
        "category":   category or rng.choice(CATEGORIES),
        "amount":     amount if amount is not None else round(rng.uniform(10, 500), 2),
    }

def random_batch(n, start):
    return [make_event(start + timedelta(seconds=rng.randint(0, 59))) for _ in range(n)]

event_schema = T.StructType([
    T.StructField("event_id",   T.StringType()),
    T.StructField("event_time", T.TimestampType()),
    T.StructField("order_id",   T.IntegerType()),
    T.StructField("region",     T.StringType()),
    T.StructField("category",   T.StringType()),
    T.StructField("amount",     T.DoubleType()),
])

orders_stream = (
    spark.readStream
    .schema(event_schema)
    .option("maxFilesPerTrigger", 1)       # no máximo 1 arquivo por micro-batch (para visualizarmos)
    .json(INPUT)
)

### 4.1 Agregação acumulada com `complete`

Receita total por região desde o início. A cada micro-batch o resultado inteiro é reescrito.

`processAllAvailable()` bloqueia até que todos os arquivos disponíveis sejam processados. É ótimo para aulas e testes, porque torna o resultado determinístico. Em produção a consulta simplesmente roda sem parar.

In [ ]:
revenue_by_region = (
    orders_stream
    .groupBy("region")
    .agg(F.count("*").alias("orders"), F.round(F.sum("amount"), 2).alias("revenue"))
)

q_total = (
    revenue_by_region.writeStream
    .format("memory")
    .queryName("revenue_by_region")
    .outputMode("complete")
    .start()
)

t0 = datetime(2024, 7, 1, 10, 0, 0)
for i in range(3):
    n = emit(random_batch(50, t0 + timedelta(minutes=i)))
    q_total.processAllAvailable()
    total = spark.sql("SELECT SUM(orders) FROM revenue_by_region").first()[0]
    print(f"Arquivo {i + 1}: +{n} eventos -> total acumulado = {total}")

spark.sql("SELECT * FROM revenue_by_region ORDER BY revenue DESC").show()
q_total.stop()

### 4.2 `update` com `foreachBatch`: o que muda em cada micro-batch

`foreachBatch` entrega cada micro-batch como um **DataFrame comum** para uma função sua. Dentro dela vale tudo da API batch: `show`, `write` em qualquer formato, `MERGE` em Delta, chamadas a APIs.

Com `update`, cada micro-batch contém **só as regiões que mudaram**.

> Evite o sink `console` em notebooks: ele imprime na saída do processo Java, que não aparece na célula.

In [ ]:
def show_batch(batch_df, batch_id):
    print(f"--- micro-batch {batch_id}: {batch_df.count()} linhas alteradas")
    batch_df.orderBy("region").show()

q_update = (
    revenue_by_region.writeStream
    .foreachBatch(show_batch)
    .outputMode("update")
    .option("checkpointLocation", f"{BASE}/chk_update")
    .start()
)
q_update.processAllAvailable()     # processa os 3 arquivos que já existem

# Um arquivo novo só com eventos da região South: só ela aparece no próximo micro-batch
emit([make_event(t0, region="South") for _ in range(10)])
q_update.processAllAvailable()
q_update.stop()

---
## 5. Janelas de tempo e watermark

### Janelas

`F.window(coluna_de_tempo, duração, deslocamento)` agrupa eventos por intervalos de **event time**.

```
  Tumbling (5 min, sem sobreposição)       Sliding (10 min, a cada 5 min)
  |10:00-10:05|10:05-10:10|10:10-10:15|    |10:00-------10:10|
                                                 |10:05-------10:15|
  cada evento cai em 1 janela                          |10:10-------10:20|
                                           cada evento cai em 2 janelas
```

### Watermark

Uma janela pode receber eventos atrasados a qualquer momento. Se o Spark guardar o estado de todas as janelas para sempre, a memória cresce sem limite. O **watermark** define por quanto tempo esperar:

```
  watermark = maior event_time já visto − atraso tolerado

  Exemplo com atraso tolerado de 5 minutos:
  maior event_time visto = 10:11   ->  watermark = 10:06

  - janela 10:00-10:05  -> fim (10:05) < watermark: fechada, é emitida e o estado é liberado
  - evento de 10:07     -> janela 10:05-10:10 ainda aberta: aceito
  - evento de 10:02     -> a janela dele já fechou: descartado
```

Dois detalhes importantes:
- Em agregações por janela, o que decide se um evento atrasado entra é **se a janela dele ainda está aberta**, e não a hora do evento comparada ao watermark.
- O watermark é recalculado **ao fim de cada micro-batch** e vale a partir do seguinte.

Com `outputMode("append")`, uma janela só é escrita **quando o watermark passa do fim dela**, porque a partir daí o resultado não muda mais.

In [ ]:
INPUT_W = f"{BASE}/input_windows"
os.makedirs(INPUT_W)

windowed = (
    spark.readStream.schema(event_schema).option("maxFilesPerTrigger", 1).json(INPUT_W)
    .withWatermark("event_time", "5 minutes")
    .groupBy(F.window("event_time", "5 minutes"))
    .agg(F.count("*").alias("orders"), F.round(F.sum("amount"), 2).alias("revenue"))
    .select(F.date_format("window.start", "HH:mm").alias("start"),
            F.date_format("window.end", "HH:mm").alias("end"), "orders", "revenue")
)

q_win = (
    windowed.writeStream
    .format("memory")
    .queryName("windows")
    .outputMode("append")
    .start()
)

def at(hh_mm):
    return datetime.strptime(f"2024-07-01 {hh_mm}", "%Y-%m-%d %H:%M")

last_batch = [-1]

def step(label, events):
    emit(events, INPUT_W)
    q_win.processAllAvailable()
    # um lote pode gerar mais de um micro-batch (ex.: um batch extra só para fechar janelas)
    new = [p for p in q_win.recentProgress if p["batchId"] > last_batch[0]]
    last_batch[0] = q_win.lastProgress["batchId"]
    dropped = sum(p["stateOperators"][0]["numRowsDroppedByWatermark"] for p in new if p["stateOperators"])
    wm = q_win.lastProgress["eventTime"].get("watermark", "-")[11:16]
    print(f"{label:<40} watermark = {wm} | descartados = {dropped}")
    spark.sql("SELECT * FROM windows ORDER BY start").show()

In [ ]:
# Lote 1: eventos entre 10:00 e 10:04. Nenhuma janela pode ser fechada ainda.
step("Lote 1: 10:00-10:04", [make_event(at(f"10:0{m}")) for m in range(5)])

In [ ]:
# Lote 2: eventos de 10:05 a 10:09 + um evento ATRASADO de 10:01 (ainda dentro do watermark)
step("Lote 2: 10:05-10:09 + atrasado 10:01",
     [make_event(at(f"10:0{m}")) for m in range(5, 10)] + [make_event(at("10:01"))])

In [ ]:
# Lote 3: eventos às 10:16 + um atrasado de 10:03.
# O watermark em vigor é 10:04, mas a janela 10:00-10:05 ainda não fechou: o atrasado entra.
step("Lote 3: 10:16 + atrasado 10:03",
     [make_event(at("10:16")) for _ in range(3)] + [make_event(at("10:03"))])

In [ ]:
# Lote 4: um evento MUITO atrasado de 10:02 (sua janela já foi fechada) + um evento de 10:12
step("Lote 4: muito atrasado 10:02 + 10:12", [make_event(at("10:02")), make_event(at("10:12"))])

Leitura do resultado:

| Lote | Watermark em vigor | O que acontece |
|------|-------------------|----------------|
| 1 | nenhum | Nada é emitido: a janela 10:00-10:05 ainda pode receber eventos |
| 2 | 09:59 | O atrasado de 10:01 é aceito. O watermark passa a 10:04, ainda antes de 10:05 |
| 3 | 10:04 | O atrasado de 10:03 é aceito, porque a janela dele segue aberta. Com o evento de 10:16, o watermark vai a 10:11 e as janelas 10:00-10:05 (7 pedidos) e 10:05-10:10 (5 pedidos) são **fechadas e emitidas** |
| 4 | 10:11 | O evento de 10:02 é **descartado**: sua janela já foi emitida. O de 10:12 entra na janela 10:10-10:15, que fica aberta |

**O trade-off do watermark:** um atraso tolerado maior aceita mais eventos atrasados, mas segura mais estado em memória e atrasa a emissão dos resultados.

In [ ]:
q_win.stop()

---
## 6. Escrevendo no data lake com checkpoint

Em produção, o sink é um sistema durável: arquivos Parquet, Delta ou Kafka. O **checkpoint** guarda:

```
  checkpoint/
  ├── offsets/    <- o que cada micro-batch vai ler   (escrito ANTES de processar)
  ├── commits/    <- quais micro-batches terminaram   (escrito DEPOIS de escrever no sink)
  ├── sources/    <- arquivos já vistos pela fonte
  └── state/      <- estado de agregações e janelas
```

Se a consulta cair, ao reiniciar com o **mesmo checkpoint** ela continua de onde parou: não perde nem duplica dados. Com um sink que também é transacional (arquivos do Spark, Delta), a garantia é **exactly-once**.

> Cada consulta precisa do seu próprio checkpoint. Apagar o checkpoint faz a consulta reprocessar tudo desde o início.

In [ ]:
SINK = f"{BASE}/lake/orders_bronze"
CHK  = f"{BASE}/checkpoints/orders_bronze"

def start_ingestion():
    return (
        spark.readStream.schema(event_schema).json(INPUT)
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("event_date", F.to_date("event_time"))
        .writeStream
        .format("parquet")
        .option("path", SINK)
        .option("checkpointLocation", CHK)
        .partitionBy("event_date")
        .outputMode("append")
        .start()
    )

q_ing = start_ingestion()
q_ing.processAllAvailable()
print("Linhas no lake:", spark.read.parquet(SINK).count())

q_ing.stop()     # simula uma queda
print("Consulta parada.")

In [ ]:
# Enquanto a consulta estava parada, chegaram mais 2 arquivos
emit(random_batch(40, t0)); emit(random_batch(40, t0))
files_in_input = len([f for f in os.listdir(INPUT) if f.endswith(".json")])
print("Arquivos na pasta de entrada:", files_in_input)

q_ing = start_ingestion()     # mesmo checkpoint
q_ing.processAllAvailable()

n_lake   = spark.read.parquet(SINK).count()
n_unique = spark.read.parquet(SINK).select("event_id").distinct().count()
print(f"Linhas no lake: {n_lake} | event_id distintos: {n_unique}  (sem duplicatas)")
print("Micro-batches confirmados:", sorted(f for f in os.listdir(f"{CHK}/commits") if not f.startswith(".")))
q_ing.stop()

---
## 7. Triggers: quando cada micro-batch roda

| Trigger | Comportamento | Uso |
|---------|---------------|-----|
| (padrão) | Novo micro-batch assim que o anterior termina | Menor latência |
| `processingTime="1 minute"` | Um micro-batch a cada intervalo | Controlar custo e tamanho dos arquivos |
| `availableNow=True` | Processa **tudo que está disponível** e **para** | Ingestão incremental agendada |
| `continuous="1 second"` | Processamento contínuo, sem micro-batch (experimental) | Latência de milissegundos, operações simples |

### `availableNow`: streaming como ingestão incremental

É um dos usos mais comuns em data lakes: roda como um job batch agendado (por exemplo, a cada hora), mas, graças ao checkpoint, **processa só os arquivos novos**. Resolve o problema da Bronze incremental da aula 5 sem controle manual de "quais arquivos já li".

In [ ]:
def incremental_run():
    q = (
        spark.readStream.schema(event_schema).json(INPUT)
        .writeStream.format("parquet")
        .option("path", f"{BASE}/lake/incremental")
        .option("checkpointLocation", f"{BASE}/checkpoints/incremental")
        .trigger(availableNow=True)
        .start()
    )
    q.awaitTermination()          # termina sozinha quando não há mais nada
    return q.lastProgress["numInputRows"] if q.lastProgress else 0

print("Execução 1 — linhas lidas:", incremental_run())
print("Execução 2 — linhas lidas:", incremental_run(), "(nada novo)")
emit(random_batch(25, t0))
print("Execução 3 — linhas lidas:", incremental_run(), "(só o arquivo novo)")
print("Total no destino:", spark.read.parquet(f"{BASE}/lake/incremental").count())

> `lastProgress` mostra só o último micro-batch. Se uma execução processar vários, some `q.recentProgress`.

---
## 8. Outras operações em streaming

| Operação | Suporte | Observação |
|----------|---------|------------|
| `select`, `filter`, `withColumn` | Sim | Iguais ao batch |
| Join stream + tabela estática | Sim | Enriquecimento com dimensões |
| Join stream + stream | Sim | Exige watermark nos dois lados e condição de tempo |
| `dropDuplicatesWithinWatermark` | Sim (3.5+) | Deduplicação com estado limitado |
| `orderBy` | Só após agregação em `complete` | Ordenar um fluxo infinito não faz sentido |
| `limit`, `count()` direto | Não | Use agregações |
| `show()`, `collect()` | Não | Use um sink (memory, foreachBatch) |

In [ ]:
# Tentar uma ação batch em um DataFrame de streaming gera erro
try:
    orders_stream.show()
except Exception as e:
    print(type(e).__name__, "->", str(e).splitlines()[0][:120])

---
## 9. Exercícios

> Use `emit(eventos, pasta)`, `make_event()`, `at()` e `processAllAvailable()` para criar cenários determinísticos. Para cada exercício, use uma pasta de entrada nova e nomes de consulta diferentes.

In [ ]:
# ── Exercício 1 ─────────────────────────────────────────────────────────────
# Conte pedidos por categoria em JANELAS DESLIZANTES de 10 minutos que avançam a cada 5 minutos.
# Use outputMode "complete" e sink memory. Envie eventos às 10:02, 10:07 e 10:12 e explique
# por que cada evento aparece em duas janelas.

# Sua solução:
EX = f"{BASE}/ex1"; os.makedirs(EX, exist_ok=True)

sliding = (
    spark.readStream.schema(event_schema).json(EX)
    .groupBy(F.window("event_time", "10 minutes", "5 minutes"), "category")
    .count()
    .select(F.date_format("window.start", "HH:mm").alias("start"),
            F.date_format("window.end", "HH:mm").alias("end"), "category", "count")
)
q1 = sliding.writeStream.format("memory").queryName("ex1_sliding").outputMode("complete").start()

emit([make_event(at(t), category="Audio") for t in ["10:02", "10:07", "10:12"]], EX)
q1.processAllAvailable()
spark.sql("SELECT * FROM ex1_sliding ORDER BY start").show()
q1.stop()
# Com janela de 10 min e passo de 5 min, todo instante pertence a 2 janelas.
# Ex.: 10:07 está em [10:00, 10:10) e em [10:05, 10:15).

In [ ]:
# ── Exercício 2 ─────────────────────────────────────────────────────────────
# Join stream-estático: enriqueça os eventos com a tabela abaixo e calcule,
# em modo "complete", a receita por "segment".

segments = spark.createDataFrame(
    [("Computers", "Hardware"), ("Peripherals", "Hardware"), ("Storage", "Hardware"),
     ("Audio", "Accessories"), ("Cables", "Accessories")],
    ["category", "segment"],
)

# Sua solução:
EX = f"{BASE}/ex2"; os.makedirs(EX, exist_ok=True)

by_segment = (
    spark.readStream.schema(event_schema).json(EX)
    .join(F.broadcast(segments), "category", "left")
    .groupBy("segment")
    .agg(F.count("*").alias("orders"), F.round(F.sum("amount"), 2).alias("revenue"))
)
q2 = by_segment.writeStream.format("memory").queryName("ex2_segments").outputMode("complete").start()

emit(random_batch(100, t0), EX)
q2.processAllAvailable()
spark.sql("SELECT * FROM ex2_segments").show()
q2.stop()

In [ ]:
# ── Exercício 3 ─────────────────────────────────────────────────────────────
# O sistema de origem às vezes reenvia o mesmo evento (mesmo event_id).
# Construa um stream que remova duplicatas usando dropDuplicatesWithinWatermark
# (watermark de 10 minutos) e grave o resultado em um sink memory em modo "append".
# Prove que, ao enviar o mesmo evento 3 vezes, ele aparece uma só vez.

# Sua solução:
EX = f"{BASE}/ex3"; os.makedirs(EX, exist_ok=True)

dedup = (
    spark.readStream.schema(event_schema).json(EX)
    .withWatermark("event_time", "10 minutes")
    .dropDuplicatesWithinWatermark(["event_id"])
)
q3 = dedup.writeStream.format("memory").queryName("ex3_dedup").outputMode("append").start()

e = make_event(at("10:00"))
emit([e], EX); emit([e, make_event(at("10:01"))], EX); emit([e], EX)     # o mesmo evento em 3 arquivos
q3.processAllAvailable()

spark.sql("SELECT event_id, event_time, COUNT(*) AS vezes FROM ex3_dedup GROUP BY event_id, event_time").show(truncate=False)
q3.stop()

---
## Resumo

| Conceito | API |
|----------|-----|
| Ler um stream | `spark.readStream.format(...).schema(...).load(...)` |
| Escrever | `df.writeStream.format(...).outputMode(...).start()` |
| Tolerância a falhas | `.option("checkpointLocation", path)` |
| Janela de tempo | `F.window("event_time", "5 minutes", "1 minute")` |
| Dados atrasados | `.withWatermark("event_time", "10 minutes")` |
| Lógica livre por micro-batch | `.foreachBatch(func)` |
| Ingestão incremental | `.trigger(availableNow=True)` |
| Monitorar | `q.status`, `q.lastProgress`, `spark.streams.active` |
| Parar | `q.stop()` |

| Output mode | Agregação sem watermark | Agregação com watermark | Sem agregação |
|-------------|------------------------|-------------------------|---------------|
| `append` | Não suportado | Janelas fechadas | Sim |
| `update` | Sim | Sim | Sim |
| `complete` | Sim | Sim (estado não é limpo) | Não suportado |

In [ ]:
for q in spark.streams.active:
    q.stop()
spark.stop()